# Concatenaci?n de CSVs de triatl?n

En este notebook se reproduce un flujo parecido al del repositorio `ndjo-byte/Triathlon_EDA`: partir de varios CSVs individuales, leerlos con `pandas`, concatenarlos y guardar un CSV final combinado.

Se generan tres archivos nuevos:

- `Data/ironman_2015_2025/ironman_wc_2015_2025_all_results_from_parts.csv`
- `Data/ironman_2015_2025/ironman_wc_2015_2025_pro_results_from_parts.csv`
- `Data/olimpica_2016_2025/world_triathlon_olympic_standard_2016_2025_all_results_from_parts.csv`

In [ ]:
import pandas as pd
from pathlib import Path

## 1. Rutas del proyecto

La variable `BASE_DIR` apunta a la ra?z del repositorio. Desde ah? se construyen las rutas de datos.

In [ ]:
BASE_DIR = Path.cwd()

# Si ejecutas el notebook desde la carpeta Notebook, subimos un nivel hasta la ra?z del repo.
if BASE_DIR.name.lower() == "notebook":
    BASE_DIR = BASE_DIR.parent

DATA_DIR = BASE_DIR / "Data"
IRONMAN_DIR = DATA_DIR / "ironman_2015_2025"
OLYMPIC_DIR = DATA_DIR / "olimpica_2016_2025"

print("Base del proyecto:", BASE_DIR)
print("Carpeta Ironman:", IRONMAN_DIR)
print("Carpeta Ol?mpica:", OLYMPIC_DIR)

## 2. Funci?n auxiliar para cargar y concatenar CSVs

Esta funci?n recibe una lista de archivos, los lee uno a uno y a?ade una columna `source_file` para poder saber de qu? CSV viene cada fila.

In [ ]:
def read_and_concat_csvs(csv_files):
    dataframes = []

    for csv_file in csv_files:
        df_temp = pd.read_csv(csv_file)
        df_temp["source_file"] = csv_file.name
        dataframes.append(df_temp)
        print(f"{csv_file.name}: {df_temp.shape[0]} filas, {df_temp.shape[1]} columnas")

    combined = pd.concat(dataframes, ignore_index=True)
    return combined

## 3. Concatenaci?n de resultados Ironman

Aqu? se leen los CSVs individuales de Ironman por a?o y g?nero. Se excluyen los archivos ya combinados y el resumen, porque queremos reconstruir el CSV total desde las partes.

In [ ]:
ironman_csv_files = sorted([
    file for file in IRONMAN_DIR.glob("ironman_wc_*_results.csv")
    if "all_results" not in file.name
    and "subevents_summary" not in file.name
])

print("N?mero de CSVs individuales de Ironman:", len(ironman_csv_files))
ironman_csv_files[:5]

In [ ]:
ironman_all = read_and_concat_csvs(ironman_csv_files)

print("Shape final Ironman:", ironman_all.shape)
ironman_all.head()

### Guardar CSV combinado de Ironman

In [ ]:
ironman_output_path = IRONMAN_DIR / "ironman_wc_2015_2025_all_results_from_parts.csv"
ironman_all.to_csv(ironman_output_path, index=False, encoding="utf-8-sig")

print("CSV Ironman combinado guardado en:")
print(ironman_output_path)

## 4. Filtrar Ironman solo profesionales

Para comparar con distancia ol?mpica, conviene usar solo profesionales. En Ironman se identifican con la columna `age_group`:

- `MPRO`: hombres profesionales
- `FPRO`: mujeres profesionales

In [ ]:
print(ironman_all.columns.tolist())

In [ ]:
ironman_all["age_group"].value_counts().head(20)

In [ ]:
ironman_pro = ironman_all[ironman_all["age_group"].isin(["MPRO", "FPRO"])].copy()

print("Shape Ironman PRO:", ironman_pro.shape)
ironman_pro["age_group"].value_counts()

In [ ]:
ironman_pro_output_path = IRONMAN_DIR / "ironman_wc_2015_2025_pro_results_from_parts.csv"
ironman_pro.to_csv(ironman_pro_output_path, index=False, encoding="utf-8-sig")

print("CSV Ironman PRO guardado en:")
print(ironman_pro_output_path)

## 5. Concatenaci?n de resultados de distancia ol?mpica

Se leen los CSVs individuales de Juegos Ol?mpicos y finales WTCS. Tambi?n se excluyen los CSVs ya combinados y el resumen.

In [ ]:
olympic_csv_files = sorted([
    file for file in OLYMPIC_DIR.glob("*.csv")
    if "all_results" not in file.name
    and "summary" not in file.name
])

print("N?mero de CSVs individuales de distancia ol?mpica:", len(olympic_csv_files))
olympic_csv_files[:5]

In [ ]:
olympic_all = read_and_concat_csvs(olympic_csv_files)

print("Shape final distancia ol?mpica:", olympic_all.shape)
olympic_all.head()

### Guardar CSV combinado de distancia ol?mpica

In [ ]:
olympic_output_path = OLYMPIC_DIR / "world_triathlon_olympic_standard_2016_2025_all_results_from_parts.csv"
olympic_all.to_csv(olympic_output_path, index=False, encoding="utf-8-sig")

print("CSV distancia ol?mpica combinado guardado en:")
print(olympic_output_path)

## 6. Comprobaciones r?pidas

Resumen de filas por dataset, a?o y g?nero.

In [ ]:
ironman_all.groupby(["championship_year", "gender"]).size()

In [ ]:
olympic_all.groupby(["dataset", "championship_year", "gender"]).size()

## 7. Conclusi?n

Este notebook deja documentado el proceso de creaci?n de los CSVs combinados desde los CSVs individuales, igual que en el proyecto de referencia.

Para el EDA comparativo, el archivo m?s recomendable es el de Ironman PRO:

`ironman_wc_2015_2025_pro_results_from_parts.csv`

porque la distancia ol?mpica contiene atletas ?lite, no grupos de edad amateurs.